# 1. Настройка окружения

**Цель:** Проверить установку PyTorch, определить доступные устройства (CPU/GPU/MPS), настроить логирование.

---

In [1]:
# Импортируем модуль os для работы с файловой системой и sys для системных параметров
import os, sys


In [2]:
# Импортируем PyTorch — главный фреймворк для глубокого обучения
import torch
# Выводим версию, чтобы убедиться, что библиотека установлена и работает
print(f"PyTorch version: {torch.__version__}")


PyTorch version: 2.11.0+cu128


In [3]:
# Импортируем модуль platform для получения информации об операционной системе
import platform

# Выводим версию Python и характеристики системы
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")


Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Processor: AMD64 Family 25 Model 97 Stepping 2, AuthenticAMD


In [4]:
# Проверка CUDA
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: CUDA not available — either no NVIDIA GPU or no CUDA toolkit")

CUDA available: True
CUDA device: NVIDIA GeForce RTX 5070 Ti
CUDA version: 12.8


In [5]:
# Проверка MPS (Apple Silicon) — ускорение на чипах Apple M1/M2/M3
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()
print(f"MPS available: {mps_available}")
print(f"MPS built: {mps_built}")
if not mps_available:
    print("MPS not available — используем CPU")

MPS available: False
MPS built: False
MPS not available — используем CPU


In [6]:
# Выбор устройства: приоритет — CUDA (NVIDIA), затем MPS (Apple), затем CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Selected device: {device}")


Selected device: cuda


In [7]:
# Тестовый тензор: создаём случайную матрицу 3×3 на выбранном устройстве
# torch.randn генерирует значения из стандартного нормального распределения N(0,1)
x = torch.randn(3, 3, device=device)
# Атрибуты тензора: форма, устройство, тип данных
print(f"Test tensor shape: {x.shape}")
print(f"Test tensor device: {x.device}")
print(f"Test tensor dtype: {x.dtype}")
print(x)


Test tensor shape: torch.Size([3, 3])
Test tensor device: cuda:0
Test tensor dtype: torch.float32
tensor([[-0.1813,  1.1461,  1.1542],
        [-0.5315, -0.6648,  1.2590],
        [-1.0095, -0.3058,  0.3146]], device='cuda:0')


## Бенчмарк производительности

Измеряем скорость умножения матриц на выбранном устройстве. Это ключевая операция в трансформерах (внимание — это матричное умножение).

In [8]:
# Производительность: базовый benchmark умножения матриц разного размера
import time

sizes = [100, 1000, 5000]
for n in sizes:
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    
    # Прогрев (warmup): выполняем несколько итераций для стабилизации скорости
    for _ in range(5):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    
    # Замер: усредняем время 20 итераций для точности
    start = time.perf_counter()
    for _ in range(20):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / 20
    
    print(f"Matrix multiply {n}x{n}: {elapsed*1000:.2f} ms")


Matrix multiply 100x100: 0.02 ms
Matrix multiply 1000x1000: 0.09 ms
Matrix multiply 5000x5000: 8.47 ms


In [10]:
# Подводим итог: окружение настроено и готово к работе
print("\n=== Environment check complete ===")
print(f"Summary: PyTorch {torch.__version__} on {device}")



=== Environment check complete ===
Summary: PyTorch 2.11.0+cu128 on cuda
